In [ ]:
!pip install git+https://github.com/mpenagar/ml_course.git
from ml_course import SimpleRegressor, ComplexRegressor, ACMEClassifier

  Cloning https://github.com/mpenagar/ml_course.git to /tmp/pip-req-build-cwwqxxgj
  Running command git clone --filter=blob:none --quiet https://github.com/mpenagar/ml_course.git /tmp/pip-req-build-cwwqxxgj
  Resolved https://github.com/mpenagar/ml_course.git to commit 8dacd39dc3e6ef2f34fd4091da27934a941a56ed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ml_course: filename=ml_course-0.1.14-py3-none-any.whl size=4210 sha256=0872340e6e6a142111a5278fa3120e7f9b7d1c08bfcccda095bae54708cdc46c
  Stored in directory: /tmp/pip-ephem-wheel-cache-ylwpnbuf/wheels/82/0d/c9/e26f81a2bfd0a90b97733a415623b538f42fbd3cffde03a6ef
Successfully built ml_course


**1 -** Ikaskuntza Automatikoko hasiberri batek ondoko kode hau idatzi du. Exekutatzean, emaitza oso onak
lortu dituela uste du, baina gutxienez LAU akats metodologiko larri egin ditu:

In [ ]:
df['rooms_ratio'] = df['total_rooms'] / df['households']
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())
scaler = StandardScaler()
scaler.fit(df.drop(columns=['target']))
X = pd.DataFrame(scaler.transform(df.drop(columns=['target'])))
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

for c in [5, 10, 15, 20]:
    model = ComplexRegressor(complexity=c, random_state=42)
    model.fit(X_train, y_train)
    print(c, root_mean_squared_error(y_test, model.predict(X_test)))

Identifikatu kode horretan dauden 4 akats metodologikoak.



In [ ]:
df['rooms_ratio'] = df['total_rooms'] / df['households']
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
# 1 - medianaren estimazioan ezin dira testeko datuak erabili
#df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())
bedrooms_median = X_train['bedrooms'].median()
X_train['bedrooms'] = X_train['bedrooms'].fillna(bedrooms_median)
X_test['bedrooms'] = X_test['bedrooms'].fillna(bedrooms_median)
# 2 - scaler-aren doitzean ezin dira testeko datuak erabili
scaler = StandardScaler()
#scaler.fit(df.drop(columns=['target']))
#X = pd.DataFrame(scaler.transform(df.drop(columns=['target'])))
scaler.fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train))
X_test = pd.DataFrame(scaler.transform(X_test))

# 3 - Ereduaren complexity parametroaren balioa aukeratzeko ezin dugu test-a erabili
# 4 - GridSearchCV erabiliko dugu complexity parametroa doitzeko
#for c in [5, 10, 15, 20]:
#    model = ComplexRegressor(complexity=c, random_state=42)
#    model.fit(X_train, y_train)
#    print(c, root_mean_squared_error(y_test, model.predict(X_test)))

grid = {
    'complexity' : [5, 10, 15, 20]
}

grid_search = GridSearchCV(
    estimator=ComplexRegressor(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='neg_mean_squared_error'
)
grid_search.fit(X_train, y_train)
model = grid_search.best_estimator_

print(f'{grid_search.best_params_=}')
print("train:", (-grid_search.best_score_)**0.5)
print("test:", root_mean_squared_error(y_test, model.predict(X_test)))

**2 -** Demagun etxebizitza datu-base txiki bat dugula, non 'energi_ziurtagiria' ezaugarri kategorikoak honako
balioak dituen: 'A', 'B', 'C', 'D' eta 'G' (azken hau oso gutxitan agertzen da).

In [ ]:
import pandas as pd
train_df = pd.DataFrame({'azalera': [80, 120, 95, 60, 150], 'ziurtagiria': ['B', 'A', 'C', 'G', 'B']})
test_df  = pd.DataFrame({'azalera': [70, 110, 85],
'ziurtagiria': ['B', 'C', 'F']})
train_df
#test_df

,azalera,ziurtagiria
0,80,B
1,120,A
2,95,C
3,60,G
4,150,B


1. Aplikatu pd.get_dummies() zuzenean train_df eta test_df multzoetan. Zer gertatzen da zutabe
kopuruarekin eta zutabeen izenekin?

In [ ]:
train_df2 = pd.get_dummies(train_df)

In [ ]:
test_df2 = pd.get_dummies(test_df)

In [ ]:
train_df2

,azalera,ziurtagiria_A,ziurtagiria_B,ziurtagiria_C,ziurtagiria_G
0,80,False,True,False,False
1,120,True,False,False,False
2,95,False,False,True,False
3,60,False,False,False,True
4,150,False,True,False,False


In [ ]:
test_df2

,azalera,ziurtagiria_B,ziurtagiria_C,ziurtagiria_F
0,70,True,False,False
1,110,False,True,False
2,85,False,False,True


Zer gertatuko litzateke eredu bat train_df horrekin entrenatu eta
test_df horretan .predict() egiten saiatzean?


2. Sortu CathegoryVectorizer klase bat (2. gaian ikusitako moduko transformazio klase bat) Scikit-learn
eko OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first') erabiliz.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

class CathegoryVectorizer:
    def __init__(self, cathegorical_columns):
        self.onehot_encoder = OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore',
            drop='first',
        )
        self.onehot_encoder.set_output(transform="pandas")
        self.cathegorical_columns = cathegorical_columns

    def fit(self, df):
        self.onehot_encoder.fit(df[self.cathegorical_columns])
        return self

    def transform(self, df):
        df = pd.concat([
            df.drop(columns=self.cathegorical_columns),
            self.onehot_encoder.transform(df[self.cathegorical_columns])
        ], axis=1)
        return df

3. Doitu (fit) zure kodetzailea train_df multzoarekin eta aplikatu (transform) bi multzoetan. Nola kudeatu
du 'G' kategoria (testean ez dagoena) eta 'F' kategoria berria (entrenamenduan ikusi ez zena)?

In [ ]:
transformer = CathegoryVectorizer(cathegorical_columns=['ziurtagiria'])
transformer.fit(train_df)
train_df2 = transformer.transform(train_df)
test_df2 = transformer.transform(test_df)

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [ ]:
train_df

,azalera,ziurtagiria
0,80,B
1,120,A
2,95,C
3,60,G
4,150,B


In [ ]:
train_df2

,azalera,ziurtagiria_B,ziurtagiria_C,ziurtagiria_G
0,80,1.0,0.0,0.0
1,120,0.0,0.0,0.0
2,95,0.0,1.0,0.0
3,60,0.0,0.0,1.0
4,150,1.0,0.0,0.0


In [ ]:
test_df

,azalera,ziurtagiria
0,70,B
1,110,C
2,85,F


In [ ]:
test_df2

,azalera,ziurtagiria_B,ziurtagiria_C,ziurtagiria_G
0,70,1.0,0.0,0.0
1,110,0.0,1.0,0.0
2,85,0.0,0.0,0.0
